# vs1: configuration comparisons and component matching

Separate working notebook: the existing `vs1_coverage_matching.ipynb` is preserved.
Select **match**, **order**, or **random** below. Random mode uses the 20 saved random associations and plots **mean ± sample SD**, never SEM. Every figure is implemented in this notebook.

Only saved PCA archives and source-count tables are loaded; raw recordings and trial caches are not opened. No participant or electrode–source association is recomputed. F and G (ERP shell comparisons and spatial maps) are intentionally excluded.

In [ ]:
from pathlib import Path
import os
import sys
import json
import hashlib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.ticker import MaxNLocator

# ---- Main notebook controls ----
MODE = 'random'                 # 'match', 'order', 'random'
COMPONENT_REFERENCE = 'match'   # Fixed within-configuration reference; not a participant rematch
MAX_COMPONENTS = 10
HEATMAP_COMPONENTS = 10
N_TIMECOURSES = 5
SHOW_ASSOCIATION_CURVES = False
ALLOW_MISSING_RANDOM = False    # False requires all 20 random associations
ANNOTATE_HEATMAP = True
STANDARDIZE_TIMECOURSES = True  # Unit SD for shape comparison; native scores remain stored
VARIANCE_LOG_SCALE = False
SAVE_FIGURES = False
EXPORT_TABLES = False

ROOT = Path(os.environ.get('MIB_PROJECT_ROOT', Path.cwd())).expanduser().resolve()
if not (ROOT / 'vs1_run_coverage.py').is_file():
    ROOT = ROOT / 'iEEGvsMEG'
if not (ROOT / 'vs1_run_coverage.py').is_file():
    raise FileNotFoundError('Start in the repository or iEEGvsMEG directory, or set MIB_PROJECT_ROOT.')
OUT_ROOT = Path(os.environ.get('MIB_OUTPUT_ROOT', ROOT / 'out')).expanduser().resolve()
sys.path.insert(0, str(ROOT))
from vs1_utils_component_matching import component_correlations, match_components, cumulative_matched_correlation
from vs0_run_analysis import read_config

MEG_CONFIGURATIONS = ['full_concatenated', 'full_average', 'coverage_average', 'paired_coverage', 'random_control']
DATASETS = ['iEEG', *MEG_CONFIGURATIONS]
LABELS = {'iEEG': 'iEEG', 'full_concatenated': 'Full concatenated', 'full_average': 'Full average',
          'coverage_average': 'Coverage average', 'paired_coverage': 'Paired coverage', 'random_control': 'Random control'}
COLORS = {'iEEG': '#8A4DB2', 'full_concatenated': '#005F45', 'full_average': '#00A88F',
          'coverage_average': '#31C6C4', 'paired_coverage': '#8BAA25', 'random_control': '#A9C7A0'}
STYLES = {'iEEG': '-', 'full_concatenated': '-', 'full_average': '--',
          'coverage_average': ':', 'paired_coverage': '-.', 'random_control': '--'}
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .22,
                     'axes.spines.top': False, 'axes.spines.right': False})
if MODE not in ('match', 'order', 'random'):
    raise ValueError('MODE must be match, order, or random.')
PAIRINGS = [f'random_{i:03d}' for i in range(1, 21)] if MODE == 'random' else [MODE]
FIGURE_DIR = OUT_ROOT / 'vs1_pca' / 'summary' / 'figures' / MODE
TABLE_DIR = OUT_ROOT / 'vs1_pca' / 'summary' / 'visualization_tables' / MODE
print(f'Mode: {MODE}; uncertainty: SD across associations; reference: {COMPONENT_REFERENCE}')

## Load saved results and analysis parameters

Configuration and pairing-file identities are checked against saved manifests. Historical results remain usable after a plotting-code change; this reader does not pretend that historical artifacts were produced by the current code. Required PCA diagnostics are checked explicitly. Missing diagnostics are shown as unavailable, never estimated from a truncated spectrum.

To populate the new diagnostics, rerun `vs1_pca` with `--recompute`. If the pipeline reports stale construction provenance, refresh `vs1_construct` first; keep `export_trials=false` for these condition-average analyses. Existing data caches need not be deleted.

In [ ]:
CONFIGURATIONS = {}
for analysis in ['vs1_construct', 'vs1_pca', 'vs1_manipulation', 'vs1_channels']:
    path = OUT_ROOT / analysis / 'config.json'
    if path.exists():
        CONFIGURATIONS[analysis] = read_config(path)
        print(CONFIGURATIONS[analysis]['command'])
        display(pd.json_normalize(CONFIGURATIONS[analysis]['parameters'], sep='.').T.rename(columns={0: analysis}))

# Loader and numerical summaries live here; no hidden plotting helper module.
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def checked_folder(analysis, pairing):
    folder = OUT_ROOT / analysis / pairing
    manifest_path = folder / 'manifest.json'
    if not manifest_path.is_file():
        raise FileNotFoundError(f'Missing completed {analysis}/{pairing}: {manifest_path}')
    manifest = json.loads(manifest_path.read_text())
    if manifest.get('status') != 'complete' or manifest.get('pairing') != pairing:
        raise ValueError(f'Incomplete or incorrect pairing identity: {manifest_path}')
    provenance = manifest.get('provenance', {})
    recorded = provenance.get('configs', {}).get(analysis)
    if recorded and recorded != sha256(OUT_ROOT / analysis / 'config.json'):
        raise ValueError(f'{analysis}/{pairing}: configuration changed after computation.')
    for name, expected in provenance.get('pairs', {}).items():
        path = OUT_ROOT / 'vs0_pairs' / name
        if not path.is_file() or sha256(path) != expected:
            raise ValueError(f'Saved pairing input changed or is missing: {path}')
    return folder, manifest

def load_pca(pairing):
    folder, manifest = checked_folder('vs1_pca', pairing)
    result = {}
    for name in DATASETS:
        path = folder / f'{name}.npz'
        with np.load(path, allow_pickle=False) as saved:
            # Do not decompress native-feature weights: these figures only require scores.
            result[name] = {key: saved[key].copy() for key in saved.files if key != 'weights'}
        if not {'scores', 'explained_variance', 'explained_variance_ratio', 'total_variance'} <= result[name].keys():
            raise ValueError(f'Incomplete PCA archive: {path}')
    return result

PCA = {}
missing = []
for pairing in PAIRINGS:
    try:
        PCA[pairing] = load_pca(pairing)
    except FileNotFoundError as error:
        if MODE != 'random' or not ALLOW_MISSING_RANDOM:
            raise
        missing.append(pairing)
        print(error)
if not PCA:
    raise ValueError('No completed PCA results available.')
PAIRINGS = list(PCA)
REFERENCE_PCA = PCA[COMPONENT_REFERENCE] if COMPONENT_REFERENCE in PCA else load_pca(COMPONENT_REFERENCE)

# Exact observation-axis agreement is required; there is no interpolation.
def observation_axis(record):
    times = np.asarray(record.get('times', []), dtype=float).ravel()
    mode = str(np.asarray(record.get('condition_mode', 'average')).item())
    conditions = np.asarray(record.get('conditions', [])).ravel()
    if not len(times):
        path = OUT_ROOT / 'vs1_construct' / 'shared' / 'times.npy'
        if path.exists():
            times = np.load(path, allow_pickle=False)
    n = record['scores'].shape[0]
    if not len(times):
        raise ValueError('No saved time axis; rerun PCA with the updated exporter.')
    if mode == 'average' and len(times) == n:
        return times, mode, conditions
    if mode == 'stack' and n % len(times) == 0:
        return np.tile(times, n // len(times)), mode, conditions
    raise ValueError('Saved time axis and PCA observation rows disagree.')

reference_axis = observation_axis(REFERENCE_PCA['iEEG'])
for pairing, records in [*PCA.items(), (COMPONENT_REFERENCE, REFERENCE_PCA)]:
    for name, record in records.items():
        axis = observation_axis(record)
        if axis[1] != reference_axis[1] or axis[0].shape != reference_axis[0].shape or not np.allclose(axis[0], reference_axis[0], rtol=0, atol=1e-7):
            raise ValueError(f'Temporal observation axes differ: {pairing}/{name}')
        if not np.array_equal(axis[2], reference_axis[2]):
            raise ValueError(f'Condition labels differ: {pairing}/{name}')

TIME_AXIS = reference_axis[0] if reference_axis[1] == 'average' else np.arange(len(reference_axis[0]))
TIME_LABEL = 'Time (s)' if reference_axis[1] == 'average' else 'Stacked condition/time observation'
MODE_LABEL = f'{MODE}: {len(PAIRINGS)} association(s)' + ('; mean ± SD' if MODE == 'random' else '')

def array_summary(values):
    a = np.asarray(values, dtype=float)
    valid = np.isfinite(a)
    n = valid.sum(axis=0)
    mean = np.divide(np.where(valid, a, 0.).sum(axis=0), n,
                     out=np.full(a.shape[1:], np.nan), where=n > 0)
    squared = np.where(valid, (a - mean)**2, 0.).sum(axis=0)
    sd = np.sqrt(np.divide(squared, n - 1, out=np.full(a.shape[1:], np.nan), where=n > 1))
    return mean, sd, n

def table_summary(table, keys, value='value'):
    # Each row must already be ONE estimate per association at the plotted point.
    if table.duplicated(['pairing', *keys]).any():
        raise ValueError('Average internal repetitions within each association before summarizing.')
    return table.groupby(keys, dropna=False)[value].agg(mean='mean', sd='std', n='count').reset_index()

print(MODE_LABEL)
if missing:
    print('Missing random associations:', missing)

## Component matching audit

Each configuration is aligned to its own fixed demographic-match reference using signed temporal correlations and a one-to-one assignment maximizing absolute correlation. Rank swaps and sign corrections are recorded. The same operation can be applied to weight columns without changing electrode rows.

This alignment is used for random-mode heatmaps and matched time courses. Explained-variance curves, manipulation subspaces, and cumulative correlation metrics retain their **original variance ranks**. Cumulative correlations solve a new assignment restricted to the first k PCs at each k.
The fixed-reference assignment maximizes $\sum_i |r(t_i^{\mathrm{ref}}, t_{\pi(i)})|$ over one-to-one assignments; the sign correction is $\mathrm{sign}(r)$. Original component ranks remain in the audit table.


In [ ]:
ALIGNMENT = {}
audit_frames = []
for name in DATASETS:
    reference = REFERENCE_PCA[name]['scores']
    k = min(reference.shape[1], *(PCA[p][name]['scores'].shape[1] for p in PAIRINGS))
    for pairing in PAIRINGS:
        result = match_components(reference[:, :k], PCA[pairing][name]['scores'], n_components=k)
        ALIGNMENT[pairing, name] = result
        audit_frames.append(result['audit'].assign(pairing=pairing, dataset=name, reference_pairing=COMPONENT_REFERENCE))
MATCHING_TABLE = pd.concat(audit_frames, ignore_index=True)[
    ['pairing', 'dataset', 'reference_pairing', 'reference_rank', 'original_rank', 'signed_r', 'absolute_r', 'sign_correction']]
print('Reference rank versus original matched rank (one-based):')
with pd.option_context('display.max_rows', 40):
    display(MATCHING_TABLE)
RANK_FREQUENCIES = MATCHING_TABLE.groupby(['dataset', 'reference_rank', 'original_rank']).size().rename('n_associations').reset_index()
RANK_FREQUENCIES['fraction'] = RANK_FREQUENCIES.n_associations / RANK_FREQUENCIES.groupby(['dataset', 'reference_rank']).n_associations.transform('sum')
display(RANK_FREQUENCIES)
if EXPORT_TABLES:
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    MATCHING_TABLE.to_csv(TABLE_DIR / 'component_matching.csv', index=False)
    RANK_FREQUENCIES.to_csv(TABLE_DIR / 'component_rank_frequencies.csv', index=False)

## 1. Total variance of every dataset before optional block scaling

Variance is the sum of feature variances after condition averaging/stacking and the configured preprocessing, but before optional equal-variance block scaling. MEG channel z-scoring and the iEEG multiplier remain applied. These modality-specific preprocessing units are not physically interchangeable.

In [ ]:
rows = []
for pairing in PAIRINGS:
    for name in DATASETS:
        record = PCA[pairing][name]
        rows.append(dict(pairing=pairing, dataset=name,
            value=float(record.get('pre_block_total_variance', np.nan)),
            n_features=float(record.get('n_features', np.nan))))
VARIANCE_PER_ASSOCIATION = pd.DataFrame(rows)
VARIANCE_SUMMARY = table_summary(VARIANCE_PER_ASSOCIATION, ['dataset']).set_index('dataset').reindex(DATASETS)
display(VARIANCE_SUMMARY)
fig, ax = plt.subplots(figsize=(12, 4.8), layout='constrained')
for i, name in enumerate(DATASETS):
    row = VARIANCE_SUMMARY.loc[name]
    if np.isfinite(row['mean']):
        ax.bar(i, row['mean'], color=COLORS[name], width=.65)
        if MODE == 'random' and np.isfinite(row['sd']):
            ax.errorbar(i, row['mean'], yerr=row['sd'], fmt='none', color='black', capsize=4)
        if SHOW_ASSOCIATION_CURVES and MODE == 'random':
            values = VARIANCE_PER_ASSOCIATION.query('dataset == @name').value.dropna().to_numpy()
            ax.scatter(i + np.linspace(-.13, .13, len(values)), values, color='black', alpha=.3, s=12)
    else:
        ax.text(i, .03, 'Rerun PCA', transform=ax.get_xaxis_transform(), rotation=90, ha='center')
ax.set(xticks=np.arange(len(DATASETS)), xticklabels=[LABELS[n] for n in DATASETS],
       ylabel='Sum of feature variances (preprocessed units)',
       title='Dataset variance before optional block scaling · ' + MODE_LABEL)
ax.tick_params(axis='x', rotation=20)
if VARIANCE_LOG_SCALE:
    ax.set_yscale('log')
if SAVE_FIGURES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_DIR / '01_pre_block_variance.png', dpi=180)
plt.show()

## 2. Manipulations: dimensionality, cumulative explained variance, and comparison metrics

Rows follow the original comparison layout: participant averaging; coverage restriction in averaged data; and paired coverage versus full concatenation. Effective dimensionality is the participation ratio of the **full** covariance spectrum, not the first k eigenvalues. The right-hand panels show temporal-subspace overlap and both directional fractions of top-k reference variance captured by the other subspace.
For all covariance eigenvalues $\lambda_j$, effective dimensionality is $D_{\mathrm{eff}}=(\sum_j\lambda_j)^2/\sum_j\lambda_j^2$. The PCA exporter calculates it exactly from $\mathrm{tr}(G)^2/\lVert G\rVert_F^2$, with centered observation Gram matrix $G=XX^\top$, without retaining the full eigenvector matrix.


In [ ]:
COMPARISONS = [
    ('Participant averaging', 'full_concatenated', 'full_average'),
    ('Coverage restriction in averaged data', 'full_average', 'coverage_average'),
    ('Coverage and participant contributions', 'full_concatenated', 'paired_coverage')]
manipulation_rows, dimensionality_rows, cumulative_rows = [], [], []
for pairing in PAIRINGS:
    for title, a_name, b_name in COMPARISONS:
        a, b = PCA[pairing][a_name], PCA[pairing][b_name]
        for name, record in [(a_name, a), (b_name, b)]:
            dimensionality_rows.append(dict(pairing=pairing, comparison=title, dataset=name,
                value=float(record.get('effective_dimensionality', np.nan))))
            for j, value in enumerate(np.cumsum(record['explained_variance_ratio'])[:MAX_COMPONENTS], 1):
                cumulative_rows.append(dict(pairing=pairing, comparison=title, dataset=name, k=j, value=value))
        maximum = min(MAX_COMPONENTS, a['scores'].shape[1], b['scores'].shape[1])
        ua = a['scores'] / np.linalg.norm(a['scores'], axis=0)
        ub = b['scores'] / np.linalg.norm(b['scores'], axis=0)
        for k in range(1, maximum + 1):
            cross = ua[:, :k].T @ ub[:, :k]
            squares = cross**2
            va, vb = a['explained_variance'][:k], b['explained_variance'][:k]
            values = {'Temporal overlap': squares.sum()/k,
                      'A variance captured by B': np.sum(squares * va[:, None])/va.sum(),
                      'B variance captured by A': np.sum(squares * vb[None, :])/vb.sum()}
            for metric, value in values.items():
                manipulation_rows.append(dict(pairing=pairing, comparison=title, metric=metric, k=k, value=np.clip(value, 0, 1)))
MANIPULATION_METRICS = pd.DataFrame(manipulation_rows)
DIMENSIONALITY_SUMMARY = table_summary(pd.DataFrame(dimensionality_rows), ['comparison', 'dataset'])
CUMULATIVE_SUMMARY = table_summary(pd.DataFrame(cumulative_rows), ['comparison', 'dataset', 'k'])
MANIPULATION_SUMMARY = table_summary(MANIPULATION_METRICS, ['comparison', 'metric', 'k'])
display(DIMENSIONALITY_SUMMARY)
display(MANIPULATION_SUMMARY)
fig, axes = plt.subplots(3, 3, figsize=(16, 10), squeeze=False, layout='constrained')
for row, (title, a_name, b_name) in enumerate(COMPARISONS):
    ax = axes[row, 0]
    for i, name in enumerate([a_name, b_name]):
        record = DIMENSIONALITY_SUMMARY.query('comparison == @title and dataset == @name').iloc[0]
        if np.isfinite(record['mean']):
            ax.bar(i, record['mean'], color=COLORS[name], width=.65)
            if MODE == 'random' and np.isfinite(record['sd']):
                ax.errorbar(i, record['mean'], yerr=record['sd'], fmt='none', color='black', capsize=3)
        else:
            ax.text(i, .05, 'Rerun PCA', transform=ax.get_xaxis_transform(), rotation=90, ha='center')
    ax.set(xticks=[0, 1], xticklabels=['A: ' + LABELS[a_name], 'B: ' + LABELS[b_name]],
           ylabel='Participation ratio', title=title + '\nEffective dimensionality')
    ax.tick_params(axis='x', labelsize=9)
    ax = axes[row, 1]
    for name in [a_name, b_name]:
        table = CUMULATIVE_SUMMARY.query('comparison == @title and dataset == @name').sort_values('k')
        x, mean, sd = table.k.to_numpy(), table['mean'].to_numpy(), table.sd.to_numpy()
        ax.plot(x, mean, label=LABELS[name], color=COLORS[name])
        if MODE == 'random':
            ax.fill_between(x, mean-sd, mean+sd, color=COLORS[name], alpha=.16)
    ax.set(xlabel='Number of original PCs', ylabel='Explained-variance fraction', title='Cumulative explained variance', ylim=(0, 1.02))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend(fontsize=8)
    ax = axes[row, 2]
    for metric, style in [('Temporal overlap', '-'), ('A variance captured by B', '--'), ('B variance captured by A', ':')]:
        table = MANIPULATION_SUMMARY.query('comparison == @title and metric == @metric').sort_values('k')
        x, mean, sd = table.k.to_numpy(), table['mean'].to_numpy(), table.sd.to_numpy()
        ax.plot(x, mean, linestyle=style, color='black', label=metric)
        if MODE == 'random':
            ax.fill_between(x, mean-sd, mean+sd, color='black', alpha=.07)
    ax.set(xlabel='Number of original PCs', ylabel='Fraction / overlap', title='Manipulation metrics', ylim=(0, 1.02))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend(fontsize=7)
fig.suptitle('MEG configuration manipulations · ' + MODE_LABEL)
if SAVE_FIGURES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_DIR / '02_manipulations.png', dpi=180)
plt.show()

## 3. Source-count effect relative to full-concatenated MEG

The first panel reports the variance of the leading reference PCs captured by the subset subspace; it is not the total variance of the subset. Internal random source-addition paths are averaged **within each participant association first**. The plotted SD is then calculated across associations. A source count available for only one association has no estimable across-association SD; support is reported in the table.

In [ ]:
source_count_tables = []
for pairing in PAIRINGS:
    folder, _ = checked_folder('vs1_channels', pairing)
    table = pd.read_csv(folder / 'metrics.csv')
    # One estimate per association/source count, retaining both requested metrics.
    within = table.groupby('n_channels')[['variance_fraction_retained', 'subspace_overlap']].mean().reset_index()
    source_count_tables.append(within.assign(pairing=pairing))
SOURCE_COUNT_PER_ASSOCIATION = pd.concat(source_count_tables, ignore_index=True)
long = SOURCE_COUNT_PER_ASSOCIATION.melt(id_vars=['pairing', 'n_channels'], var_name='metric', value_name='value')
SOURCE_COUNT_SUMMARY = table_summary(long, ['metric', 'n_channels'])
display(SOURCE_COUNT_SUMMARY)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout='constrained')
for ax, metric, title in zip(axes, ['variance_fraction_retained', 'subspace_overlap'],
                             ['Reference variance captured', 'Temporal-subspace overlap']):
    table = SOURCE_COUNT_SUMMARY.query('metric == @metric').sort_values('n_channels')
    x, mean, sd = table.n_channels.to_numpy(), table['mean'].to_numpy(), table.sd.to_numpy()
    if SHOW_ASSOCIATION_CURVES and MODE == 'random':
        for _, individual in SOURCE_COUNT_PER_ASSOCIATION.groupby('pairing'):
            individual = individual.sort_values('n_channels')
            ax.plot(individual.n_channels, individual[metric], color=COLORS['full_concatenated'], alpha=.12, linewidth=.7)
    ax.plot(x, mean, color=COLORS['full_concatenated'], marker='.', label='Subset vs full concatenation')
    if MODE == 'random':
        ax.fill_between(x, mean-sd, mean+sd, color=COLORS['full_concatenated'], alpha=.18)
    ax.set(xlabel='Number of MEG sources', ylabel='Fraction', title=title, ylim=(0, 1.02))
    ax.legend(fontsize=8)
fig.suptitle('Source-count effect · ' + MODE_LABEL)
if SAVE_FIGURES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_DIR / '03_source_count.png', dpi=180)
plt.show()

## 4. Paired coverage versus control: cumulative component correlations

One figure, two panels: correlation with full-concatenated MEG and correlation with iEEG. At every k, match **only the first k original variance-ranked components** of the two representations and report their mean absolute temporal correlation. This is separate from alignment to the fixed reference used for time-course aggregation. The comparison uses the same participant association on both sides.

In [ ]:
correlation_rows = []
for pairing in PAIRINGS:
    for target in ['full_concatenated', 'iEEG']:
        for name in ['paired_coverage', 'random_control']:
            table = cumulative_matched_correlation(PCA[pairing][target]['scores'], PCA[pairing][name]['scores'], MAX_COMPONENTS)
            correlation_rows.append(table.rename(columns={'mean_absolute_r': 'value'}).assign(pairing=pairing, target=target, dataset=name))
CUMULATIVE_CORRELATIONS = pd.concat(correlation_rows, ignore_index=True)
CUMULATIVE_CORRELATION_SUMMARY = table_summary(CUMULATIVE_CORRELATIONS, ['target', 'dataset', 'k'])
display(CUMULATIVE_CORRELATION_SUMMARY)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout='constrained')
for ax, target in zip(axes, ['full_concatenated', 'iEEG']):
    for name in ['paired_coverage', 'random_control']:
        table = CUMULATIVE_CORRELATION_SUMMARY.query('target == @target and dataset == @name').sort_values('k')
        x, mean, sd = table.k.to_numpy(), table['mean'].to_numpy(), table.sd.to_numpy()
        ax.plot(x, mean, label=LABELS[name], color=COLORS[name], linestyle=STYLES[name], marker='.')
        if MODE == 'random':
            ax.fill_between(x, mean-sd, mean+sd, color=COLORS[name], alpha=.18)
    ax.set(xlabel='Cumulative number of original PCs (k)', ylabel='Mean absolute matched temporal r',
           title='Correlation with ' + LABELS[target], ylim=(0, 1.02))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend(fontsize=9)
fig.suptitle('Paired coverage versus saved control · ' + MODE_LABEL)
if SAVE_FIGURES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_DIR / '04_paired_control_correlations.png', dpi=180)
plt.show()

## 5. Absolute MEG–iEEG component-correlation heatmaps

One figure per MEG configuration. Rows are that configuration's fixed reference component ranks; columns are iEEG reference ranks. Compute **absolute correlations within each association before averaging**, rather than correlating the averaged time courses. Random mode adds a separate SD heatmap. Mean panels share a 0–1 scale; SD panels share a scale across configurations. A strong diagonal is not imposed by matching MEG components to iEEG: alignment uses each configuration's own fixed reference.

In [ ]:
HEATMAPS = {}
heatmap_summary_rows = []
for name in MEG_CONFIGURATIONS:
    k = min(HEATMAP_COMPONENTS, *(ALIGNMENT[p, name]['scores'].shape[1] for p in PAIRINGS),
            *(ALIGNMENT[p, 'iEEG']['scores'].shape[1] for p in PAIRINGS))
    matrices = [np.abs(component_correlations(ALIGNMENT[p, name]['scores'][:, :k],
                                             ALIGNMENT[p, 'iEEG']['scores'][:, :k])) for p in PAIRINGS]
    mean, sd, n = array_summary(matrices)
    HEATMAPS[name] = dict(mean=mean, sd=sd, n=n)
    for i in range(k):
        for j in range(k):
            heatmap_summary_rows.append(dict(dataset=name, meg_reference_rank=i+1, ieeg_reference_rank=j+1,
                                             mean=mean[i, j], sd=sd[i, j], n=n[i, j]))
HEATMAP_SUMMARY = pd.DataFrame(heatmap_summary_rows)
display(HEATMAP_SUMMARY)
finite_sd = np.concatenate([v['sd'][np.isfinite(v['sd'])] for v in HEATMAPS.values()])
sd_limit = float(finite_sd.max()) if len(finite_sd) else 0.
sd_limit = max(sd_limit, .01)  # Visual scale floor; avoid magnifying roundoff (~1e-16).
for name in MEG_CONFIGURATIONS:
    panels = [('mean', 'Mean absolute r' if MODE == 'random' else 'Absolute r', 1., 'viridis')]
    if MODE == 'random' and len(PAIRINGS) > 1:
        panels.append(('sd', 'SD of absolute r', sd_limit, 'magma'))
    fig, axes = plt.subplots(1, len(panels), figsize=(5.4*len(panels), 4.8), squeeze=False, layout='constrained')
    for ax, (key, title, vmax, cmap) in zip(axes.flat, panels):
        values = HEATMAPS[name][key]
        image = ax.imshow(np.ma.masked_invalid(values), vmin=0, vmax=vmax, cmap=cmap, origin='upper', aspect='equal')
        k = len(values)
        ax.set(xticks=np.arange(k), yticks=np.arange(k), xticklabels=np.arange(1, k+1), yticklabels=np.arange(1, k+1),
               xlabel='iEEG reference rank', ylabel=LABELS[name] + ' reference rank', title=title)
        ax.grid(False)
        fig.colorbar(image, ax=ax, shrink=.8)
        if ANNOTATE_HEATMAP and k <= 10:
            for i in range(k):
                for j in range(k):
                    value = values[i, j]
                    if np.isfinite(value):
                        rgba = image.cmap(image.norm(value))
                        luminance = .2126*rgba[0] + .7152*rgba[1] + .0722*rgba[2]
                        ax.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=8,
                                color='black' if luminance > .5 else 'white')
    fig.suptitle(LABELS[name] + ' vs iEEG · ' + MODE_LABEL)
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIGURE_DIR / f'05_correlations_{name}.png', dpi=180)
    plt.show()

## 6. First five time courses before and after component matching

One figure per configuration, including iEEG: rows are the first five components; left columns retain original ranks/signs and right columns use reference ranks/signs. In random mode both columns show the association mean ± SD. The after-matching panel overlays the corresponding fixed reference as a dashed gray line. Standardization, if selected, is for displaying temporal shape only; it does not modify saved scores or PCA variance.

In [ ]:
TIMECOURSE_SUMMARIES = {}
for name in DATASETS:
    k = min(N_TIMECOURSES, *(PCA[p][name]['scores'].shape[1] for p in PAIRINGS),
            *(ALIGNMENT[p, name]['scores'].shape[1] for p in PAIRINGS))
    raw = np.stack([PCA[p][name]['scores'][:, :k] for p in PAIRINGS])
    aligned = np.stack([ALIGNMENT[p, name]['scores'][:, :k] for p in PAIRINGS])
    reference = REFERENCE_PCA[name]['scores'][:, :k].copy()
    if STANDARDIZE_TIMECOURSES:
        raw = (raw - raw.mean(axis=1, keepdims=True)) / raw.std(axis=1, keepdims=True, ddof=1)
        aligned = (aligned - aligned.mean(axis=1, keepdims=True)) / aligned.std(axis=1, keepdims=True, ddof=1)
        reference = (reference - reference.mean(axis=0)) / reference.std(axis=0, ddof=1)
    raw_mean, raw_sd, raw_n = array_summary(raw)
    aligned_mean, aligned_sd, aligned_n = array_summary(aligned)
    TIMECOURSE_SUMMARIES[name] = dict(raw_mean=raw_mean, raw_sd=raw_sd, raw_n=raw_n,
        aligned_mean=aligned_mean, aligned_sd=aligned_sd, aligned_n=aligned_n)
    fig, axes = plt.subplots(k, 2, figsize=(13, 2.15*k), squeeze=False, sharex=True, sharey='row', layout='constrained')
    for component in range(k):
        for col, (mean, sd, curves) in enumerate([(raw_mean, raw_sd, raw), (aligned_mean, aligned_sd, aligned)]):
            ax = axes[component, col]
            if SHOW_ASSOCIATION_CURVES and MODE == 'random':
                ax.plot(TIME_AXIS, curves[:, :, component].T, color=COLORS[name], alpha=.1, linewidth=.6)
            ax.plot(TIME_AXIS, mean[:, component], color=COLORS[name], label=LABELS[name])
            if MODE == 'random':
                ax.fill_between(TIME_AXIS, mean[:, component]-sd[:, component], mean[:, component]+sd[:, component], color=COLORS[name], alpha=.18)
            if col == 1:
                ax.plot(TIME_AXIS, reference[:, component], '--', color='gray', linewidth=1, label=COMPONENT_REFERENCE + ' reference')
            if reference_axis[1] == 'average' and TIME_AXIS.min() <= 0 <= TIME_AXIS.max():
                ax.axvline(0, color='gray', linewidth=.6)
            ax.axhline(0, color='gray', linewidth=.5, alpha=.5)
            ax.set(title=f'{"Original PC" if col == 0 else "Reference component"} {component+1}',
                   ylabel='Score (unit SD)' if STANDARDIZE_TIMECOURSES else 'PCA score')
            if component == k-1:
                ax.set_xlabel(TIME_LABEL)
    axes[0, 0].set_title('Before matching · original PC 1')
    axes[0, 1].set_title('After matching · reference component 1')
    axes[0, 1].legend(fontsize=8)
    fig.suptitle(LABELS[name] + ' · first ' + str(k) + ' component time courses · ' + MODE_LABEL)
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIGURE_DIR / f'06_timecourses_{name}.png', dpi=180)
    plt.show()

## 7. Explained-variance ratio: iEEG and all MEG configurations on one panel

Original variance ranks are retained. Component matching never reorders this curve. SD represents variation across saved participant associations; identical iEEG preprocessing and participants should yield no meaningful iEEG variation.

In [ ]:
variance_ratio_rows = []
for pairing in PAIRINGS:
    for name in DATASETS:
        values = PCA[pairing][name]['explained_variance_ratio'][:MAX_COMPONENTS]
        variance_ratio_rows.extend(dict(pairing=pairing, dataset=name, k=i+1, value=100*v) for i, v in enumerate(values))
EXPLAINED_VARIANCE_SUMMARY = table_summary(pd.DataFrame(variance_ratio_rows), ['dataset', 'k'])
display(EXPLAINED_VARIANCE_SUMMARY)
fig, ax = plt.subplots(figsize=(12, 5), layout='constrained')
for name in DATASETS:
    table = EXPLAINED_VARIANCE_SUMMARY.query('dataset == @name').sort_values('k')
    x, mean, sd = table.k.to_numpy(), table['mean'].to_numpy(), table.sd.to_numpy()
    ax.plot(x, mean, label=LABELS[name], color=COLORS[name], linestyle=STYLES[name], marker='.')
    if MODE == 'random':
        ax.fill_between(x, mean-sd, mean+sd, color=COLORS[name], alpha=.15)
ax.set(xlabel='Original variance-ranked PC', ylabel='Explained-variance ratio (%)',
       title='Explained variance · all configurations and iEEG · ' + MODE_LABEL)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend(ncol=2, fontsize=9)
if SAVE_FIGURES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_DIR / '07_explained_variance.png', dpi=180)
plt.show()

if EXPORT_TABLES:
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    for name, table in {'pre_block_variance': VARIANCE_SUMMARY.reset_index(),
                        'effective_dimensionality': DIMENSIONALITY_SUMMARY,
                        'manipulations': MANIPULATION_SUMMARY, 'source_count': SOURCE_COUNT_SUMMARY,
                        'cumulative_correlations': CUMULATIVE_CORRELATION_SUMMARY,
                        'heatmap_correlations': HEATMAP_SUMMARY,
                        'explained_variance': EXPLAINED_VARIANCE_SUMMARY}.items():
        table.to_csv(TABLE_DIR / f'{name}.csv', index=False)